# CatBoost enriquecido: vendedor, atributos y texto

## Qué aprende

CatBoost construye árboles en secuencia: cada nuevo árbol corrige errores de los anteriores. Para este problema usamos el `seller_id`, categoría, tipo de publicación, precio/stock/ventas, logística y atributos del producto como categóricas. Las combinaciones vendedor×categoría y vendedor×tipo permiten capturar políticas distintas de publicación.

El título y los valores de atributos también se convierten en TF-IDF y se comprimen con TruncatedSVD a 72 números densos. SVD conserva combinaciones frecuentes de palabras; no es una red neuronal ni reemplaza el texto original.

## Por qué encaja y límites

- **A favor:** CatBoost maneja categóricas de cardinalidad alta y captura relaciones no lineales entre categoría, precio, vendedor y stock.
- **En contra:** tarda más que modelos lineales; los IDs de vendedor pueden no ayudar en vendedores nuevos. Las features de venta/stock describen el snapshot del aviso, así que la interpretación es “clasificar esta publicación con los campos entregados”, no inferir condición antes de que se publique.

## Resultado experimental

Accuracy **0,9276**, balanced accuracy **0,9280**, macro F1 **0,9273**; holdout agrupado por `parent_item_id` de 21.000 filas. El mismo holdout se consultó en los 10 intentos: el valor sirve para seleccionar candidatos y puede estar optimista. El objetivo 0,94 no se alcanzó.

```mermaid
flowchart TD
 A["Título + atributos"] --> B["TF-IDF + SVD (72)"]
 C["Vendedor × categoría"] --> E["CatBoost"]
 D["Precio, stock, ventas, logística"] --> E
 B --> E
 E --> F["new / used"]
```


In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
for p in (ROOT,*ROOT.parents):
    if (p/'src'/'ap2026q2_common.py').exists(): ROOT=p; break
sys.path.insert(0,str(ROOT/'src'))
from ap2026q2_common import *
from ap2026q2_advanced import build_advanced_features, add_text_svd
from sklearn.model_selection import StratifiedGroupKFold
from catboost import CatBoostClassifier
print('Repo:',ROOT,'| datos:',project_data_dir())

Repo: C:\Users\arira\OneDrive\Documents\GitHub\tp_ap | datos: C:\Users\arira\OneDrive\Documents\TP 3 AP


In [ ]:
raw_train, raw_test, sample = load_raw_data()
y = raw_train['condition'].astype(str).reset_index(drop=True)
groups = raw_train['parent_item_id'].astype('string').fillna('__ROW__' + raw_train['id'].astype(str)).to_numpy()
folds = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42).split(np.zeros(len(y)), y, groups))
valid_idx = np.concatenate([folds[i][1] for i in (0, 1, 2)])
fit_idx = np.setdiff1d(np.arange(len(y)), valid_idx)
assert not set(groups[fit_idx]).intersection(set(groups[valid_idx]))
print('Desarrollo:',len(fit_idx),'| holdout:',len(valid_idx),'| familias compartidas:',len(set(groups[fit_idx]) & set(groups[valid_idx])))

In [ ]:
# Seleccionamos IDs de atributos usando solo el desarrollo.
x_fit, attribute_ids = build_advanced_features(raw_train.iloc[fit_idx].reset_index(drop=True), top_k=75)
x_valid, _ = build_advanced_features(raw_train.iloc[valid_idx].reset_index(drop=True), attribute_ids=attribute_ids)
x_fit, x_valid, tfidf, svd = add_text_svd(x_fit, x_valid, n_components=72, random_state=42)
cat_cols = x_fit.select_dtypes(include=['object','string','category']).columns.tolist()
for col in cat_cols:
    x_fit[col] = x_fit[col].astype('string').fillna('__MISSING__').astype(str)
    x_valid[col] = x_valid[col].astype('string').fillna('__MISSING__').astype(str)
print('Features:',x_fit.shape[1],'| atributos:',len(attribute_ids),'| SVD varianza:',round(svd.explained_variance_ratio_.sum(),4))

In [ ]:
MODEL_PARAMS = dict(iterations=850, depth=7, learning_rate=.04, loss_function='Logloss',
                    eval_metric='Accuracy', l2_leaf_reg=8, random_seed=51,
                    verbose=100, thread_count=-1, allow_writing_files=False,
                    early_stopping_rounds=85, one_hot_max_size=10, random_strength=.5)
model = CatBoostClassifier(**MODEL_PARAMS)
model.fit(x_fit, y.iloc[fit_idx], cat_features=cat_cols,
          eval_set=(x_valid, y.iloc[valid_idx]), verbose=100)
pred = model.predict(x_valid).reshape(-1)
metrics = score_predictions(y.iloc[valid_idx], pred)
display(pd.Series(metrics, name='holdout por familia'))
print('Best iteration:',model.best_iteration_)

## Crear submission

Esta celda vuelve a ajustar en las 70.000 etiquetas. Si Kaggle no publica `sample_submission.csv`, puede inferir un template seguro desde los 30.000 IDs de `test`: conserva esos IDs y usa los encabezados `ID,condition`.

Si luego aparece un template oficial, conviene usarlo y dejar `USE_INFERRED_TEMPLATE = False`.

In [ ]:
GENERATE_SUBMISSION = True
USE_INFERRED_TEMPLATE = True
if GENERATE_SUBMISSION:
    train_full, test_full, sample = load_raw_data()
    if sample is None and USE_INFERRED_TEMPLATE:
        create_submission_template_from_test()
    y_full = train_full['condition'].astype(str)
    x_full, ids_attr = build_advanced_features(train_full, top_k=75)
    x_test, _ = build_advanced_features(test_full, attribute_ids=ids_attr)
    x_full, x_test, tfidf_full, svd_full = add_text_svd(x_full, x_test, n_components=72, random_state=42)
    cats_full = x_full.select_dtypes(include=['object','string','category']).columns.tolist()
    for col in cats_full:
        x_full[col] = x_full[col].astype('string').fillna('__MISSING__').astype(str)
        x_test[col] = x_test[col].astype('string').fillna('__MISSING__').astype(str)
    full_params = {key:value for key,value in MODEL_PARAMS.items() if key != 'early_stopping_rounds'}
    final_model = CatBoostClassifier(**{**full_params, 'iterations': 490, 'verbose': False})
    final_model.fit(x_full, y_full, cat_features=cats_full, verbose=False)
    output = write_submission('catboost_advanced_svd', final_model.predict(x_test).reshape(-1), test_full['id'].astype('string'))
    print(output)